In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



In [36]:
# read the file the excel
# all panda data frames are called by df
file_path = r"C:\Users\ADMIN\OneDrive\Documents\Agriculture_Data_Mining.xlsx"

df = pd.read_excel(file_path, sheet_name="3_Agriculture_Business_Data")


print(df.tail())
print(df.head())
print(df.shape)

df.info() # the colm names and the data types
print(df.isna().sum().sort_values(ascending=False)) # find missing values under each column.
print(df.nunique())  # Find the number of unique values in each column
print(df.describe(include ="all"))
duplicate_count = df.duplicated().sum() #findng the sum of the number of duplicates in the dataset
print("Exact duplicates()",duplicate_count)
df= df.drop_duplicates().reset_index(drop=True)# removes identical duplicates in every column
farm_id_counts = df["Farm_ID"].value_counts()

# where there is a near-duplicate
repeated_farms = farm_id_counts[farm_id_counts > 1]
print(repeated_farms)

repeated_ids = repeated_farms.index
print(df[df["Farm_ID"].isin(repeated_ids)].sort_values("Farm_ID"))


# check whether the same records represent different observations
print(df[df["Farm_ID"].isin(repeated_ids)].groupby("Farm_ID")[["Year", "Month"]].nunique())

#remove extra spaces and standardize capitalization
text_columns = df.select_dtypes(include="object").columns
for column in text_columns:
    df[column] = df[column].str.strip() # remove extar space witht the strip word

# columns where capitalization is constant
for column in ["Region", "Crop", "Farm_Type", "Soil_Type","Irrigation_Method", "Market_Channel", "Season"]:
    if column in df.columns:
        df[column] = df[column].str.title()# the title keyword make them capital

# yes or no clumns remove the spaces and capitalize
yes_no_columns = ["Organic_Certified","Quality_Certified","Uses_Digital_Marketing"]
for column in yes_no_columns:
    if column in df.columns:
        df[column] = df[column].str.strip().str.title()

# fix the dtypes an d convert numeric columns
numeric_columns = ["Soil_pH","Customer_Rating","Avg_Payment_Days","Farm_Loan_KES","Training_Hours","Distance_to_Market_km",
    "Employees",
    "Customer_Count",
    "Year",
    "Month"
]
for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# covert categories to categorical data
category_columns = ["Region","Crop","Farm_Type","Soil_Type","Irrigation_Method","Market_Channel", "Season"]
for column in category_columns:
    if column in df.columns:
        df[column] = df[column].astype("category")


# conver whole-number to intergers
integer_columns = ["Year","Month","Employees", "Customer_Count","Avg_Payment_Days"]
for column in integer_columns:
    if column in df.columns:
        df[column] = df[column].astype("Int64")

#check inavlid and missing values
df.loc[df["Customer_Rating"].between(1, 5),"Customer_Rating"] = np.nan

# check missing values
missing_summary = (df.isna().sum().sort_values(ascending=False))
print(missing_summary[missing_summary > 0])

# filling the missing numerical values with median
numeric_columns = df.select_dtypes(include="number").columns
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# filling missing categorical values with mode
category_columns = df.select_dtypes(include=["object","category"]).columns
for column in category_columns:
    if df[column].isna().any():
        mode_value =df[column].mode()
        if not mode_value.empty:
           df[column] =df[column].fillna (mode_value.iloc[0])


#no replacing meangfu blanks
df["Storage_Type"] = df["Storage_Type"].fillna("No Storage")
    



       Farm_ID    Region      Crop    Farm_Type Soil_Type Irrigation_Method  \
12195  F107890  Western   Potatoes  Smallholder     Sandy           Rainfed   
12196  F101768   Eastern       Tea  Smallholder     Sandy           Rainfed   
12197  F109929  Western     Coffee  Smallholder     Sandy           Rainfed   
12198  F103240   Western     Maize  Smallholder      Clay           Rainfed   
12199  F107203   Central    Coffee  Smallholder     Sandy         Sprinkler   

      Market_Channel       Season Organic_Certified Quality_Certified  ...  \
12195         Export          Dry                No                No  ...   
12196         Export   Long Rains                No                No  ...   
12197      Wholesale  Short Rains                No                No  ...   
12198          Local          Dry                No                No  ...   
12199      Wholesale   Long Rains               Yes                No  ...   

      Customer_Count Customer_Rating  Avg_Payment_Days  

In [35]:
#finaaly clean dataset
print(df.shape)
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head())
print("Exact duplicates:", df.duplicated().sum())

(12000, 47)
Farm_ID                       object
Region                        object
Crop                          object
Farm_Type                     object
Soil_Type                     object
Irrigation_Method             object
Market_Channel                object
Season                        object
Organic_Certified             object
Quality_Certified             object
Storage_Type                  object
Sales_Channel                 object
Farm_Area_Ha                 float64
Annual_Rainfall_mm           float64
Avg_Temperature_C            float64
Avg_Humidity_pct             float64
Soil_pH                      float64
Nitrogen_kg_ha               float64
Phosphorus_kg_ha             float64
Potassium_kg_ha              float64
Fertilizer_kg                float64
Pesticide_L                  float64
Seed_Cost_KES                float64
Labor_Cost_KES               float64
Machinery_Cost_KES           float64
Irrigation_Cost_KES          float64
Transport_Cost_KES        